In [1]:
import torch
import os
import copy
import sys

In [2]:
# Add project path
sys.path.append(os.path.abspath("../data"))
from models.gcn_model import GCN

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
def fedavg(ldp_models):
    """
    Aggregates LDP-protected local models into a global model update
    using Federated Averaging (FedAvg).

    Args:
        ldp_models (list): List of LDP-trained PyTorch models

    Returns:
        global_model (torch.nn.Module): Aggregated global model
    """
    # Initialize global model with same architecture
    global_model = copy.deepcopy(ldp_models[0])
    global_state = global_model.state_dict()

    # Parameter-wise averaging
    for key in global_state.keys():
        global_state[key] = torch.mean(
            torch.stack([model.state_dict()[key].float() for model in ldp_models]),
            dim=0
        )

    global_model.load_state_dict(global_state)
    return global_model

In [5]:
if __name__ == "__main__":

    # Load one graph ONLY to get input dimension (no training data used)
    graph = torch.load("../data/graph/graph_A.pt", weights_only=False)
    input_dim = graph.num_node_features

    # Initialize models (architecture must match)
    model_A = GCN(input_dim, 32, 2).to(device)
    model_B = GCN(input_dim, 32, 2).to(device)
    model_C = GCN(input_dim, 32, 2).to(device)

    # Load LDP-protected local model updates
    model_A.load_state_dict(torch.load("../data/models/model_A_ldp.pth"))
    model_B.load_state_dict(torch.load("../data/models/model_B_ldp.pth"))
    model_C.load_state_dict(torch.load("../data/models/model_C_ldp.pth"))

    print("LDP-protected local models loaded")

    # Aggregate local models → global model update
    global_model = fedavg([model_A, model_B, model_C])

    # Save global update
    os.makedirs("../data/models", exist_ok=True)
    torch.save(global_model.state_dict(), "../data/models/global_model.pth")

    print("Global model update created and stored successfully")

LDP-protected local models loaded
Global model update created and stored successfully
